In [11]:
!pip install pyspark findspark


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import findspark
# Tự động tìm thư viện Spark trong hệ thống
findspark.init()

from pyspark.sql import SparkSession

# 1. Khởi tạo phiên làm việc Spark
spark = SparkSession.builder \
    .appName("Football_Player_Valuation") \
    .master("local[*]") \
    .getOrCreate()

print("Đã khởi tạo SparkSession thành công!")

# 2. Trỏ đường dẫn đến file players.csv đang nằm trên HDFS
players_path = "hdfs://localhost:9820/football_project/players.csv"

# 3. Đọc file CSV thành dạng DataFrame
players_df = spark.read.csv(players_path, header=True, inferSchema=True)

# 4. Hiển thị thông tin bảng
print("\n--- Cấu trúc các cột dữ liệu (Schema) ---")
players_df.printSchema()

print("\n--- 5 dòng dữ liệu cầu thủ đầu tiên ---")
players_df.show(5, truncate=False)

Đã khởi tạo SparkSession thành công!

--- Cấu trúc các cột dữ liệu (Schema) ---
root
 |-- player_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- name: string (nullable = true)
 |-- last_season: integer (nullable = true)
 |-- current_club_id: integer (nullable = true)
 |-- player_code: string (nullable = true)
 |-- country_of_birth: string (nullable = true)
 |-- city_of_birth: string (nullable = true)
 |-- country_of_citizenship: string (nullable = true)
 |-- date_of_birth: timestamp (nullable = true)
 |-- sub_position: string (nullable = true)
 |-- position: string (nullable = true)
 |-- foot: string (nullable = true)
 |-- height_in_cm: integer (nullable = true)
 |-- contract_expiration_date: timestamp (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- international_caps: integer (nullable = true)
 |-- international_goals: integer (nullable = true)
 |-- current_n

In [13]:
from pyspark.sql.functions import col, current_date, datediff, floor
from pyspark.sql.types import IntegerType

# 1. Lọc bỏ các cầu thủ chưa có giá trị chuyển nhượng
# (Vì 'market_value_in_eur' là biến mục tiêu cần dự đoán, nếu Null thì máy không thể học được)
clean_df = players_df.dropna(subset=["market_value_in_eur"])

# 2. Trích xuất đặc trưng (Feature Engineering): Tính Tuổi (Age)
# Lấy ngày hiện tại trừ đi ngày sinh (date_of_birth), chia 365 và làm tròn xuống thành số nguyên
clean_df = clean_df.withColumn(
    "age", 
    floor(datediff(current_date(), col("date_of_birth")) / 365).cast(IntegerType())
)

# 3. Xử lý giá trị thiếu (Missing values) cho một số cột quan trọng
# Giả sử chân thuận (foot) bị thiếu, ta điền mặc định là 'right' (chiếm đa số). 
# Chiều cao thiếu thì điền trung bình khoảng 180cm
clean_df = clean_df.fillna({
    "foot": "right", 
    "height_in_cm": 180
})

# 4. Loại bỏ các cột dư thừa không giúp ích cho việc dự đoán giá (để giảm nhẹ dữ liệu)
# Lưu ý: Tuỳ vào file CSV của bạn có cột nào, nếu không có hàm drop sẽ tự bỏ qua
columns_to_drop = [
    "first_name", "last_name", "name", "player_code", 
    "city_of_birth", "url", "image_url", "date_of_birth"
]
clean_df = clean_df.drop(*columns_to_drop)

# 5. In kết quả đối chiếu
total_raw = players_df.count()
total_clean = clean_df.count()

print(f"Tổng số cầu thủ ban đầu: {total_raw}")
print(f"Tổng số cầu thủ sau khi làm sạch (có giá trị chuyển nhượng): {total_clean}")
print(f"Đã loại bỏ {total_raw - total_clean} dòng dữ liệu không hợp lệ.\n")

print("--- Dữ liệu sau khi thêm cột Tuổi (age) và làm sạch ---")
clean_df.select("player_id", "country_of_citizenship", "position", "foot", "height_in_cm", "age", "market_value_in_eur").show(5)

Tổng số cầu thủ ban đầu: 50149
Tổng số cầu thủ sau khi làm sạch (có giá trị chuyển nhượng): 41528
Đã loại bỏ 8621 dòng dữ liệu không hợp lệ.

--- Dữ liệu sau khi thêm cột Tuổi (age) và làm sạch ---
+---------+----------------------+----------+-----+------------+---+-------------------+
|player_id|country_of_citizenship|  position| foot|height_in_cm|age|market_value_in_eur|
+---------+----------------------+----------+-----+------------+---+-------------------+
|       10|               Germany|    Attack|right|         184| 48|            1000000|
|       26|               Germany|Goalkeeper| left|         190| 46|             750000|
|       65|              Bulgaria|    Attack|right|         180| 45|            1000000|
|       77|                Brazil|  Defender|right|         180| 48|             200000|
|       80|               Germany|Goalkeeper|right|         194| 45|             100000|
+---------+----------------------+----------+-----+------------+---+-------------------+
o

In [14]:
# 1. Đăng ký DataFrame thành một bảng ảo để có thể dùng SQL truy vấn
# Chúng ta sẽ đặt tên bảng này là 'football_players'
players_df.createOrReplaceTempView("football_players")

# 2. Viết câu lệnh truy vấn bằng SQL quen thuộc
# Yêu cầu: Tìm top 10 cầu thủ có giá trị chuyển nhượng (market_value_in_eur) cao nhất.
# Sắp xếp giảm dần và loại bỏ những cầu thủ bị thiếu dữ liệu giá (NULL).
query = """
    SELECT 
        player_id, 
        name, 
        current_club_id, 
        country_of_citizenship, 
        market_value_in_eur 
    FROM football_players 
    WHERE market_value_in_eur IS NOT NULL 
    ORDER BY market_value_in_eur DESC 
    LIMIT 10
"""

# 3. Thực thi truy vấn bằng spark.sql()
top_players_df = spark.sql(query)

# 4. Hiển thị kết quả ra màn hình
print("\n--- TOP 10 CẦU THỦ CÓ GIÁ TRỊ CAO NHẤT ---")
top_players_df.show(truncate=False)


--- TOP 10 CẦU THỦ CÓ GIÁ TRỊ CAO NHẤT ---


+---------+---------------------+---------------+----------------------+-------------------+
|player_id|name                 |current_club_id|country_of_citizenship|market_value_in_eur|
+---------+---------------------+---------------+----------------------+-------------------+
|418560   |Erling Haaland       |281            |Norway                |200000000          |
|937958   |Lamine Yamal         |131            |Spain                 |200000000          |
|342229   |Kylian Mbappé        |418            |France                |180000000          |
|566723   |Michael Olise        |27             |France                |150000000          |
|683840   |Pedri                |131            |Spain                 |150000000          |
|371998   |Vinicius Junior      |418            |Brazil                |140000000          |
|487469   |Vitinha              |583            |Portugal              |140000000          |
|502670   |Khvicha Kvaratskhelia|583            |Georgia              

In [17]:
!pip install numpy
!pip install pandas

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.6 MB 3.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/12.6 MB 4.5 MB/s eta 0:00:03
   --------------- ------------------------ 4.7/12.6 MB 5.9 MB/s eta 0:00:02
   -------------------- ------------------- 6.3/12.6 MB 6.0 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.6 MB 6.2 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.6 MB 6.2 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 4.8 MB/s eta 0:00:01
   -------------------------- ------------- 8.4/12.6 MB 4.7 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.6 MB 4.4 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.6 MB 4.1 MB/s eta 0:00:01
   -----------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 7.2 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 12.0 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.7 MB 12.0 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.7 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 9.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

# 1. Chuẩn bị dữ liệu: Chọn các cột cần thiết và loại bỏ dữ liệu trống (NULL)
ml_df = players_df.select(
    "name", 
    "market_value_in_eur", 
    "highest_market_value_in_eur"
).na.drop()

# 2. VectorAssembler: Đóng gói các cột chỉ số thành một cột 'features' duy nhất (bắt buộc trong Spark ML)
assembler = VectorAssembler(
    inputCols=["market_value_in_eur", "highest_market_value_in_eur"],
    outputCol="features"
)
assembled_data = assembler.transform(ml_df)

# 3. StandardScaler: Đưa các con số hàng chục triệu Euro về cùng một thang đo tỷ lệ
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scaler_model = scaler.fit(assembled_data)
scaled_data = scaler_model.transform(assembled_data)

# 4. Khởi tạo và huấn luyện mô hình K-Means (k=3 nghĩa là chia làm 3 cụm)
kmeans = KMeans(featuresCol="scaledFeatures", k=3, seed=42)
model = kmeans.fit(scaled_data)

# 5. Gán nhãn cụm (prediction) cho từng cầu thủ
predictions = model.transform(scaled_data)

# Hiển thị kết quả
print("\n--- PHÂN CỤM CẦU THỦ BẰNG K-MEANS ---")
print("Nhãn (prediction): 0, 1, 2 đại diện cho 3 đẳng cấp khác nhau")
predictions.select("name", "market_value_in_eur", "highest_market_value_in_eur", "prediction").show(15, truncate=False)


--- PHÂN CỤM CẦU THỦ BẰNG K-MEANS ---
Nhãn (prediction): 0, 1, 2 đại diện cho 3 đẳng cấp khác nhau
+---------------------+-------------------+---------------------------+----------+
|name                 |market_value_in_eur|highest_market_value_in_eur|prediction|
+---------------------+-------------------+---------------------------+----------+
|Miroslav Klose       |1000000            |30000000                   |2         |
|Roman Weidenfeller   |750000             |8000000                    |0         |
|Dimitar Berbatov     |1000000            |34500000                   |2         |
|Lúcio                |200000             |24500000                   |0         |
|Tom Starke           |100000             |3000000                    |0         |
|Dedê                 |400000             |9500000                    |0         |
|Christoph Metzelder  |1500000            |9500000                    |0         |
|Tomas Rosicky        |350000             |17500000                   

In [1]:
from kafka import KafkaProducer
import json
import time
import random

# Khởi tạo Producer kết nối với Kafka server đang chạy ở port 9092
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print("Đang mô phỏng luồng dữ liệu chuyển nhượng cầu thủ thời gian thực...\n")

# Danh sách cầu thủ giả lập để đẩy vào luồng
players = ["Kylian Mbappe", "Erling Haaland", "Jude Bellingham", "Vinicius Jr", "Phil Foden"]

# Vòng lặp bắn dữ liệu (ví dụ: 10 giao dịch)
for i in range(10):
    player = random.choice(players)
    # Random giá trị chuyển nhượng từ 80 đến 250 triệu Euro
    value = random.randint(80, 250) 
    
    # Đóng gói dữ liệu dạng JSON (rất phổ biến trong Big Data)
    transaction = {
        "player_name": player,
        "market_value_m_eur": value,
        "timestamp": int(time.time())
    }
    
    # Gửi tin nhắn vào topic 'football_transfers'
    producer.send('football_transfers', value=transaction)
    print(f"Đã phát sóng: {transaction}")
    
    # Nghỉ 2 giây giữa mỗi giao dịch để dễ quan sát luồng streaming
    time.sleep(2)

# Đảm bảo toàn bộ tin nhắn trong hàng đợi được gửi đi
producer.flush()
print("\nHoàn tất phiên mô phỏng!")

Đang mô phỏng luồng dữ liệu chuyển nhượng cầu thủ thời gian thực...

Đã phát sóng: {'player_name': 'Phil Foden', 'market_value_m_eur': 246, 'timestamp': 1790766234}
Đã phát sóng: {'player_name': 'Kylian Mbappe', 'market_value_m_eur': 184, 'timestamp': 1790766236}
Đã phát sóng: {'player_name': 'Phil Foden', 'market_value_m_eur': 223, 'timestamp': 1790766238}
Đã phát sóng: {'player_name': 'Kylian Mbappe', 'market_value_m_eur': 246, 'timestamp': 1790766240}
Đã phát sóng: {'player_name': 'Vinicius Jr', 'market_value_m_eur': 106, 'timestamp': 1790766242}
Đã phát sóng: {'player_name': 'Phil Foden', 'market_value_m_eur': 152, 'timestamp': 1790766244}
Đã phát sóng: {'player_name': 'Kylian Mbappe', 'market_value_m_eur': 213, 'timestamp': 1790766246}
Đã phát sóng: {'player_name': 'Erling Haaland', 'market_value_m_eur': 144, 'timestamp': 1790766248}
Đã phát sóng: {'player_name': 'Phil Foden', 'market_value_m_eur': 199, 'timestamp': 1790766250}
Đã phát sóng: {'player_name': 'Erling Haaland', 'mark

In [2]:
from kafka import KafkaConsumer
import json

# Khởi tạo Consumer để đọc dữ liệu từ topic 'football_transfers'
consumer = KafkaConsumer(
    'football_transfers',
    bootstrap_servers=['localhost:9092'],
    # auto_offset_reset='earliest' giúp đọc lại toàn bộ tin nhắn từ lúc bắt đầu
    auto_offset_reset='earliest', 
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    # consumer_timeout_ms=10000 giúp tự động ngắt kết nối sau 10 giây nếu không có dữ liệu mới
    consumer_timeout_ms=10000 
)

print("Đang lắng nghe luồng dữ liệu Streaming từ Kafka...\n")

# Vòng lặp liên tục đọc tin nhắn mới
for message in consumer:
    data = message.value
    print(f"Nhận được: Cầu thủ {data['player_name']} | Định giá: {data['market_value_m_eur']} triệu Euro | Thời gian: {data['timestamp']}")
    
print("\nĐã đọc hết dữ liệu hiện có trong hàng đợi!")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25696\4289916537.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Đang lắng nghe luồng dữ liệu Streaming từ Kafka...

Nhận được: Cầu thủ Phil Foden | Định giá: 246 triệu Euro | Thời gian: 1790766234
Nhận được: Cầu thủ Kylian Mbappe | Định giá: 184 triệu Euro | Thời gian: 1790766236
Nhận được: Cầu thủ Phil Foden | Định giá: 223 triệu Euro | Thời gian: 1790766238
Nhận được: Cầu thủ Kylian Mbappe | Định giá: 246 triệu Euro | Thời gian: 1790766240
Nhận được: Cầu thủ Vinicius Jr | Định giá: 106 triệu Euro | Thời gian: 1790766242
Nhận được: Cầu thủ Phil Foden | Định giá: 152 triệu Euro | Thời gian: 1790766244
Nhận được: Cầu thủ Kylian Mbappe | Định giá: 213 triệu Euro | Thời gian: 1790766246
Nhận được: Cầu thủ Erling Haaland | Định giá: 144 triệu Euro | Thời gian: 1790766248
Nhận được: Cầu thủ Phil Foden | Định giá: 199 triệu Euro | Thời gian: 1790766250
Nhận được: Cầu thủ Erling Haaland | Định giá: 232 triệu Euro | Thời gian: 1790766252

Đã đọc hết dữ liệu hiện có trong hàng đợi!
